In [0]:
dbutils.widgets.removeAll()

In [0]:
# =============================================================================
# BRONZE → SILVER - JOB REUTILIZABLE
# =============================================================================

dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "1. Ambiente")
dbutils.widgets.text("process_date", "", "2. Fecha Proceso")
dbutils.widgets.text("tabla_silver", "", "3. Tabla Silver ")

env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
tabla_silver = dbutils.widgets.get("tabla_silver").strip()

catalog = f"{env}_fraud"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
tabla_full = f"{silver_schema}.{tabla_silver}"

print(f"🚀 INICIO TRANSFORMACIÓN BRONZE → SILVER")
print(f"   Ambiente     : {env}")
print(f"   Fecha proceso: {process_date}")
print(f"   Tabla Silver : {tabla_full}\n")


In [0]:
from pyspark.sql.functions import lit, col, when, to_timestamp, concat_ws
from datetime import datetime
import pytz
ingestion_ts = datetime.now(pytz.timezone('America/Lima')).strftime('%Y-%m-%d %H:%M:%S')

# ====================== LÓGICA POR TABLA ======================
if tabla_silver == "clientes_silver":
    df = spark.table(f"{bronze_schema}.clientes_bronze")
    df = df.withColumn("edad_real", 2026 - col("ano_nacimiento")) \
           .withColumn("rango_edad", when(col("edad_real") < 30, "Joven")
                                    .when(col("edad_real") < 50, "Adulto")
                                    .otherwise("Senior")) \
           .withColumn("rango_fico", when(col("puntaje_fico") >= 750, "Excelente")
                                     .when(col("puntaje_fico") >= 650, "Bueno")
                                     .otherwise("Riesgo"))

elif tabla_silver == "tarjetas_silver":
    df = spark.table(f"{bronze_schema}.tarjetas_bronze")

elif tabla_silver == "transacciones_silver":
    df = spark.table(f"{bronze_schema}.transacciones_bronze") \
             .join(spark.table(f"{bronze_schema}.catalogo_mcc_bronze"), "codigo_mcc", "left") \
             .withColumn("fecha_completa", to_timestamp(concat_ws("-", col("ano"), col("mes"), col("dia")))) \
             .withColumn("categoria_mcc", when(col("descripcion").contains("grocery"), "Alimentación")
                                          .when(col("descripcion").contains("restaurant"), "Restaurantes")
                                          .otherwise("Otros"))

elif tabla_silver == "mcc_silver":
    df = spark.table(f"{bronze_schema}.catalogo_mcc_bronze")

else:
    raise ValueError(f"Tabla {tabla_silver} no soportada")

# Auditoría
df = df.withColumn("ingestion_timestamp", lit(ingestion_ts)) \
       .withColumn("process_date", lit(process_date))

# Escritura
df.write.format("delta").mode("overwrite").option("overwriteSchema", "false").saveAsTable(tabla_full)

registros = df.count()



In [0]:
# Resumen
print("\n" + "═" * 85)
print(" RESUMEN BRONZE → SILVER ".center(85, "═"))
print("═" * 85)
print(f"{'Tabla Silver':<25} : {tabla_full}")
print(f"{'Registros procesados':<25} : {registros:,}")
print(f"{'Timestamp Perú':<25} : {ingestion_ts}")
print("═" * 85 + "\n")

# Top 10 legible
print(f"🔍 TOP 10 de {tabla_silver}")
display(df.limit(10).toPandas().style.set_properties(**{'text-align': 'left'}))

print("\n✅ TRANSFORMACIÓN SILVER FINALIZADA")